# El "Hello World" de las Redes Neuronales: Aprendiendo la Compuerta XOR

---

**Objetivo:** Entrenar una red neuronal desde cero para aprender la función lógica XOR, transparentando cada cálculo de gradiente y actualización de pesos.

**¿Por qué XOR?** Es el ejemplo mínimo que requiere una red con **capas ocultas**, ya que los datos no son linealmente separables. Es el punto de partida histórico que motivó el desarrollo del algoritmo de **backpropagation**.

---

> *"El problema XOR demostró que una capa oculta puede representar cualquier función booleana."*  
> — Rumelhart, Hinton & Williams (1986)

## 1. El Problema XOR

La función **XOR** (OR exclusivo) produce `1` solo cuando exactamente una de sus entradas es `1`:

| $x_1$ | $x_2$ | $y = x_1 \oplus x_2$ |
|:------:|:------:|:---------------------:|
|   0    |   0    |          0            |
|   0    |   1    |          1            |
|   1    |   0    |          1            |
|   1    |   1    |          0            |

### ¿Por qué no puede aprenderlo un Perceptrón simple?

Un perceptrón calcula $\hat{y} = \sigma(w_1 x_1 + w_2 x_2 + b)$, que define un **hiperplano separador** en el espacio de entradas. Los puntos XOR **no pueden separarse con una sola línea recta** — necesitamos al menos una capa oculta.

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

# Datos XOR
X = np.array([[0, 0],
              [0, 1],
              [1, 0],
              [1, 1]], dtype=float)
Y = np.array([[0], [1], [1], [0]], dtype=float)

etiquetas = ["XOR=0", "XOR=1", "XOR=1", "XOR=0"]
colores   = ["#e74c3c" if y == 0 else "#2ecc71" for y in Y.flatten()]

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=X[Y.flatten()==0, 0], y=X[Y.flatten()==0, 1],
    mode='markers+text',
    marker=dict(size=22, color='#e74c3c', line=dict(width=2, color='#c0392b'), symbol='circle'),
    text=["(0,0) → 0", "(1,1) → 0"],
    textposition=["bottom center", "top center"],
    textfont=dict(size=13),
    name='XOR = 0'
))
fig.add_trace(go.Scatter(
    x=X[Y.flatten()==1, 0], y=X[Y.flatten()==1, 1],
    mode='markers+text',
    marker=dict(size=22, color='#2ecc71', line=dict(width=2, color='#27ae60'), symbol='square'),
    text=["(0,1) → 1", "(1,0) → 1"],
    textposition=["top center", "bottom center"],
    textfont=dict(size=13),
    name='XOR = 1'
))

# Intento de separación lineal (imposible)
x_line = np.linspace(-0.3, 1.3, 100)
fig.add_trace(go.Scatter(
    x=x_line, y=-x_line + 1,
    mode='lines', line=dict(dash='dash', color='gray', width=2),
    name='Intento lineal (falla)'
))
fig.add_annotation(x=0.85, y=0.6, text="❌ No existe<br>separación lineal",
                   showarrow=True, arrowhead=2, ax=30, ay=-30,
                   font=dict(color="gray", size=12))

fig.update_layout(
    title=dict(text="El problema XOR: no linealmente separable", font=dict(size=18)),
    xaxis=dict(title="x₁", range=[-0.4, 1.4], tickvals=[0, 1]),
    yaxis=dict(title="x₂", range=[-0.4, 1.4], tickvals=[0, 1]),
    legend=dict(x=0.01, y=0.99),
    width=600, height=480,
    plot_bgcolor='#f8f9fa'
)
fig.show()

## 2. Arquitectura de la Red Neuronal

Usaremos una red **2 → 2 → 1** (dos entradas, dos neuronas ocultas, una salida):

$$
\text{Capa de entrada} \xrightarrow{W^{(1)}, b^{(1)}} \text{Capa oculta} \xrightarrow{W^{(2)}, b^{(2)}} \text{Capa de salida}
$$

### Notación

| Símbolo | Significado |
|---------|-------------|
| $\mathbf{x} \in \mathbb{R}^2$ | Vector de entrada |
| $W^{(1)} \in \mathbb{R}^{2 \times 2}$ | Pesos capa oculta |
| $\mathbf{b}^{(1)} \in \mathbb{R}^2$ | Sesgos capa oculta |
| $W^{(2)} \in \mathbb{R}^{1 \times 2}$ | Pesos capa salida |
| $b^{(2)} \in \mathbb{R}$ | Sesgo capa salida |
| $\mathbf{h}$ | Activaciones capa oculta |
| $\hat{y}$ | Predicción de salida |

### Función de activación: Sigmoide

$$\boxed{\sigma(z) = \frac{1}{1 + e^{-z}}}$$

Su derivada tiene una forma muy conveniente:

$$\sigma'(z) = \sigma(z)\,(1 - \sigma(z))$$

> **¿Por qué sigmoide?** Comprime cualquier valor real a $(0, 1)$, ideal para modelar probabilidades. Su derivada es suave y se expresa en términos de la propia función.

In [ ]:
# Visualización de la arquitectura y la función sigmoide

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=("Arquitectura 2→2→1", "Función Sigmoide y su derivada"),
                    column_widths=[0.45, 0.55])

# --- Diagrama de red ---
nodos = {
    'x1':  (0, 1.5), 'x2':  (0, 0.5),
    'h1':  (1, 2.0), 'h2':  (1, 0.0),
    'out': (2, 1.0)
}
colores_nodo = {
    'x1': '#3498db', 'x2': '#3498db',
    'h1': '#9b59b6', 'h2': '#9b59b6',
    'out': '#e67e22'
}
etiq_nodo = {'x1': 'x₁', 'x2': 'x₂', 'h1': 'h₁', 'h2': 'h₂', 'out': 'ŷ'}
capas = {'x1': 'Entrada', 'x2': 'Entrada', 'h1': 'Oculta', 'h2': 'Oculta', 'out': 'Salida'}

conexiones = [('x1','h1'), ('x1','h2'), ('x2','h1'), ('x2','h2'), ('h1','out'), ('h2','out')]
for (a, b) in conexiones:
    xa, ya = nodos[a]; xb, yb = nodos[b]
    fig.add_trace(go.Scatter(x=[xa, xb], y=[ya, yb],
        mode='lines', line=dict(color='#bdc3c7', width=2),
        showlegend=False), row=1, col=1)

for nombre, (x, y) in nodos.items():
    fig.add_trace(go.Scatter(
        x=[x], y=[y], mode='markers+text',
        marker=dict(size=38, color=colores_nodo[nombre],
                    line=dict(width=2, color='white')),
        text=[etiq_nodo[nombre]], textfont=dict(size=14, color='white'),
        textposition='middle center',
        name=capas[nombre], showlegend=False
    ), row=1, col=1)

# Etiquetas de capas
for cx, label, color in [(0,'Entrada\n(x)','#3498db'), (1,'Oculta\n(h)','#9b59b6'), (2,'Salida\n(ŷ)','#e67e22')]:
    fig.add_annotation(x=cx, y=-0.6, text=label, showarrow=False,
                       font=dict(size=11, color=color), row=1, col=1)

# Etiquetas de pesos
fig.add_annotation(x=0.5, y=1.55, text="W⁽¹⁾, b⁽¹⁾", showarrow=False,
                   font=dict(size=11, color='#7f8c8d'), row=1, col=1)
fig.add_annotation(x=1.5, y=1.35, text="W⁽²⁾, b⁽²⁾", showarrow=False,
                   font=dict(size=11, color='#7f8c8d'), row=1, col=1)

# --- Sigmoide y derivada ---
z = np.linspace(-6, 6, 200)
sig = 1 / (1 + np.exp(-z))
dsig = sig * (1 - sig)

fig.add_trace(go.Scatter(x=z, y=sig, mode='lines',
    line=dict(color='#2980b9', width=3), name='σ(z)'), row=1, col=2)
fig.add_trace(go.Scatter(x=z, y=dsig, mode='lines',
    line=dict(color='#e74c3c', width=3, dash='dash'), name="σ'(z)"), row=1, col=2)
fig.add_vline(x=0, line_dash='dot', line_color='gray', row=1, col=2)
fig.add_hline(y=0.5, line_dash='dot', line_color='#2980b9', opacity=0.4, row=1, col=2)
fig.add_annotation(x=3, y=0.98, text="σ(z)→1", font=dict(color='#2980b9'), row=1, col=2)
fig.add_annotation(x=-3, y=0.05, text="σ(z)→0", font=dict(color='#2980b9'), row=1, col=2)
fig.add_annotation(x=0.3, y=0.27, text="máx σ'=0.25<br>en z=0", font=dict(color='#e74c3c', size=11),
                   showarrow=True, ax=40, ay=-10, row=1, col=2)

fig.update_layout(
    height=380, width=950,
    title_text="",
    legend=dict(x=0.55, y=0.95),
    plot_bgcolor='#f8f9fa',
    paper_bgcolor='white'
)
fig.update_xaxes(range=[-0.4, 2.4], showticklabels=False, showgrid=False, row=1, col=1)
fig.update_yaxes(range=[-0.9, 2.5], showticklabels=False, showgrid=False, row=1, col=1)
fig.update_xaxes(title_text="z", row=1, col=2)
fig.update_yaxes(title_text="valor", row=1, col=2)
fig.show()

## 3. Paso Hacia Adelante (Forward Pass)

El forward pass transforma la entrada $\mathbf{x}$ en una predicción $\hat{y}$ aplicando transformaciones afines y no lineales en cada capa.

### Capa oculta

**Preactivación** (combinación lineal):
$$\mathbf{z}^{(1)} = W^{(1)}\,\mathbf{x} + \mathbf{b}^{(1)}$$

donde $W^{(1)} = \begin{pmatrix} w^{(1)}_{11} & w^{(1)}_{12} \\ w^{(1)}_{21} & w^{(1)}_{22} \end{pmatrix}$, es decir, la neurona $j$ recibe $z^{(1)}_j = w^{(1)}_{j1}x_1 + w^{(1)}_{j2}x_2 + b^{(1)}_j$

**Activación** (no linealidad):
$$\mathbf{h} = \sigma(\mathbf{z}^{(1)}) = \begin{pmatrix} \sigma(z^{(1)}_1) \\ \sigma(z^{(1)}_2) \end{pmatrix}$$

### Capa de salida

**Preactivación**:
$$z^{(2)} = W^{(2)}\,\mathbf{h} + b^{(2)} = w^{(2)}_1 h_1 + w^{(2)}_2 h_2 + b^{(2)}$$

**Predicción final**:
$$\boxed{\hat{y} = \sigma(z^{(2)})}$$

### Función de pérdida: Error Cuadrático Medio

$$\boxed{\mathcal{L} = \frac{1}{2}(\hat{y} - y)^2}$$

El factor $\frac{1}{2}$ es convencional: simplifica la derivada eliminando el coeficiente $2$.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_deriv(z):
    s = sigmoid(z)
    return s * (1 - s)

def forward_pass_detallado(x, W1, b1, W2, b2, verbose=True):
    """Forward pass mostrando cada cálculo intermedio."""
    # --- Capa oculta ---
    z1 = W1 @ x + b1
    h  = sigmoid(z1)

    # --- Capa de salida ---
    z2   = W2 @ h + b2
    yhat = sigmoid(z2)

    if verbose:
        print("=" * 55)
        print(f"  ENTRADA:  x = [{x[0]:.0f}, {x[1]:.0f}]")
        print("=" * 55)
        print("\n📐 CAPA OCULTA")
        print(f"  Preactivación z₁ = W⁽¹⁾x + b⁽¹⁾")
        print(f"    z₁[0] = {W1[0,0]:.3f}·{x[0]:.0f} + {W1[0,1]:.3f}·{x[1]:.0f} + {b1[0]:.3f} = {z1[0]:.4f}")
        print(f"    z₁[1] = {W1[1,0]:.3f}·{x[0]:.0f} + {W1[1,1]:.3f}·{x[1]:.0f} + {b1[1]:.3f} = {z1[1]:.4f}")
        print(f"  Activación h = σ(z₁)")
        print(f"    h[0] = σ({z1[0]:.4f}) = {h[0]:.4f}")
        print(f"    h[1] = σ({z1[1]:.4f}) = {h[1]:.4f}")
        print("\n📐 CAPA DE SALIDA")
        print(f"  Preactivación z₂ = W⁽²⁾h + b⁽²⁾")
        print(f"    z₂ = {W2[0,0]:.3f}·{h[0]:.4f} + {W2[0,1]:.3f}·{h[1]:.4f} + {b2[0]:.3f} = {z2[0]:.4f}")
        print(f"  Predicción ŷ = σ(z₂) = σ({z2[0]:.4f}) = {yhat[0]:.4f}")
        print("=" * 55)

    return z1, h, z2, yhat

# Inicialización con semilla fija para reproducibilidad
np.random.seed(42)
W1 = np.random.randn(2, 2) * 0.5
b1 = np.zeros(2)
W2 = np.random.randn(1, 2) * 0.5
b2 = np.zeros(1)

print("Pesos iniciales:")
print(f"  W⁽¹⁾ = \n{W1}")
print(f"  b⁽¹⁾ = {b1}")
print(f"  W⁽²⁾ = {W2}")
print(f"  b⁽²⁾ = {b2}")
print("\n--- Forward pass con la primera muestra (0, 0) → esperado: 0 ---")
z1, h, z2, yhat = forward_pass_detallado(X[0], W1, b1, W2, b2)

## 4. Retropropagación del Error (Backpropagation)

El objetivo es calcular $\frac{\partial \mathcal{L}}{\partial W}$ para cada parámetro y actualizar usando **descenso de gradiente**:

$$\theta \leftarrow \theta - \eta \frac{\partial \mathcal{L}}{\partial \theta}$$

Aplicamos la **regla de la cadena** propagando el error desde la salida hacia la entrada.

---

### 4.1 Gradiente en la capa de salida

Comenzamos con la pérdida:
$$\mathcal{L} = \frac{1}{2}(\hat{y} - y)^2$$

**Paso 1** — Derivada respecto a $\hat{y}$:
$$\frac{\partial \mathcal{L}}{\partial \hat{y}} = \hat{y} - y$$

**Paso 2** — Derivada respecto a la preactivación $z^{(2)}$ (regla de la cadena):
$$\delta^{(2)} \equiv \frac{\partial \mathcal{L}}{\partial z^{(2)}} = \frac{\partial \mathcal{L}}{\partial \hat{y}} \cdot \frac{\partial \hat{y}}{\partial z^{(2)}} = (\hat{y} - y)\,\sigma'(z^{(2)}) = (\hat{y} - y)\,\hat{y}(1-\hat{y})$$

**Paso 3** — Gradientes de los pesos de la capa de salida:
$$\frac{\partial \mathcal{L}}{\partial W^{(2)}} = \delta^{(2)} \cdot \mathbf{h}^\top \qquad \frac{\partial \mathcal{L}}{\partial b^{(2)}} = \delta^{(2)}$$

---

### 4.2 Propagación hacia la capa oculta

**Paso 4** — Propagar el error hacia $\mathbf{h}$:
$$\frac{\partial \mathcal{L}}{\partial \mathbf{h}} = \left(W^{(2)}\right)^\top \delta^{(2)}$$

**Paso 5** — Derivada respecto a la preactivación oculta $\mathbf{z}^{(1)}$:
$$\delta^{(1)} \equiv \frac{\partial \mathcal{L}}{\partial \mathbf{z}^{(1)}} = \frac{\partial \mathcal{L}}{\partial \mathbf{h}} \odot \sigma'(\mathbf{z}^{(1)}) = \frac{\partial \mathcal{L}}{\partial \mathbf{h}} \odot \mathbf{h} \odot (1 - \mathbf{h})$$

donde $\odot$ denota el producto elemento a elemento (Hadamard).

**Paso 6** — Gradientes de los pesos de la capa oculta:
$$\frac{\partial \mathcal{L}}{\partial W^{(1)}} = \delta^{(1)} \cdot \mathbf{x}^\top \qquad \frac{\partial \mathcal{L}}{\partial \mathbf{b}^{(1)}} = \delta^{(1)}$$

---

### Resumen del flujo de gradientes

$$\mathcal{L} \xrightarrow{\partial/\partial \hat{y}} \delta^{(2)} \xrightarrow{\partial/\partial \mathbf{h}} \delta^{(1)} \xrightarrow{\partial/\partial \mathbf{x}} \text{(no necesario)}$$

> **Intuición:** El error "fluye hacia atrás" por la red, con cada capa recibiendo su porción de responsabilidad ponderada por sus pesos.

In [ ]:
def backward_pass_detallado(x, y, z1, h, z2, yhat, W2, verbose=True):
    """Backpropagation mostrando cada gradiente intermedio."""

    # --- CAPA DE SALIDA ---
    # Paso 1: dL/d(yhat)
    dL_dyhat = yhat - y                           # escalar

    # Paso 2: delta de salida = dL/dz2
    d_sigma_z2 = yhat * (1 - yhat)               # σ'(z2)
    delta2 = dL_dyhat * d_sigma_z2               # escalar

    # Paso 3: gradientes W2 y b2
    dW2 = delta2 * h.reshape(1, -1)              # (1,2)
    db2 = delta2                                  # escalar

    # --- CAPA OCULTA ---
    # Paso 4: propagar error a h
    dL_dh = W2.T * delta2                        # (2,1)

    # Paso 5: delta de capa oculta = dL/dz1
    d_sigma_z1 = h * (1 - h)                     # σ'(z1), shape (2,)
    delta1 = dL_dh.flatten() * d_sigma_z1        # (2,)

    # Paso 6: gradientes W1 y b1
    dW1 = np.outer(delta1, x)                    # (2,2)
    db1 = delta1                                  # (2,)

    if verbose:
        y_scalar   = float(y.flat[0])
        yhat_scalar = float(yhat.flat[0])
        loss = 0.5 * (yhat_scalar - y_scalar) ** 2
        print("=" * 55)
        print(f"  BACKPROP: x={[int(xi) for xi in x]}, y={int(y_scalar)}, ŷ={yhat_scalar:.4f}")
        print(f"  Pérdida L = ½(ŷ-y)² = {loss:.4f}")
        print("=" * 55)
        print("\n⬅ CAPA DE SALIDA")
        print(f"  Paso 1 — dL/dŷ = ŷ - y = {yhat_scalar:.4f} - {int(y_scalar)} = {float(dL_dyhat.flat[0]):.4f}")
        print(f"  Paso 2 — σ'(z₂) = ŷ(1-ŷ) = {float(d_sigma_z2.flat[0]):.4f}")
        print(f"           δ² = (ŷ-y)·σ'(z₂) = {float(dL_dyhat.flat[0]):.4f}·{float(d_sigma_z2.flat[0]):.4f} = {float(delta2.flat[0]):.4f}")
        print(f"  Paso 3 — ∂L/∂W⁽²⁾ = δ²·hᵀ = {dW2}")
        print(f"           ∂L/∂b⁽²⁾ = δ² = {float(db2.flat[0]):.4f}")
        print("\n⬅ CAPA OCULTA")
        print(f"  Paso 4 — ∂L/∂h = (W⁽²⁾)ᵀ·δ² = {dL_dh.flatten()}")
        print(f"  Paso 5 — σ'(z₁) = h⊙(1-h) = {d_sigma_z1}")
        print(f"           δ¹ = (∂L/∂h) ⊙ σ'(z₁) = {delta1}")
        print(f"  Paso 6 — ∂L/∂W⁽¹⁾ = δ¹·xᵀ =\n{dW1}")
        print(f"           ∂L/∂b⁽¹⁾ = δ¹ = {db1}")
        print("=" * 55)

    return dW1, db1, dW2, db2

# Demostración con la muestra (0,1) → esperado: 1
print("--- Backprop con la muestra (0,1) → esperado: 1 ---\n")
x_demo, y_demo = X[1], Y[1]
z1_d, h_d, z2_d, yhat_d = forward_pass_detallado(x_demo, W1, b1, W2, b2, verbose=False)
dW1_d, db1_d, dW2_d, db2_d = backward_pass_detallado(
    x_demo, y_demo, z1_d, h_d, z2_d, yhat_d, W2)

## 5. Bucle de Entrenamiento

El entrenamiento itera por **épocas**. En cada época procesamos todas las muestras (modo *online* / estocástico):

1. **Forward pass** → obtener $\hat{y}$
2. **Calcular pérdida** $\mathcal{L}$
3. **Backward pass** → obtener todos los $\frac{\partial \mathcal{L}}{\partial \theta}$
4. **Actualizar pesos**: $\theta \leftarrow \theta - \eta \cdot \frac{\partial \mathcal{L}}{\partial \theta}$

La **tasa de aprendizaje** $\eta$ controla el tamaño del paso. Si es muy grande, oscilamos; si es muy pequeña, convergemos lento.

### Actualización de pesos — ecuaciones completas

$$W^{(2)} \leftarrow W^{(2)} - \eta\,\delta^{(2)}\,\mathbf{h}^\top$$

$$b^{(2)} \leftarrow b^{(2)} - \eta\,\delta^{(2)}$$

$$W^{(1)} \leftarrow W^{(1)} - \eta\,\delta^{(1)}\,\mathbf{x}^\top$$

$$\mathbf{b}^{(1)} \leftarrow \mathbf{b}^{(1)} - \eta\,\delta^{(1)}$$

In [ ]:
def entrenar(X, Y, epocas=10000, lr=0.5, seed=42, guardar_cada=50):
    """
    Entrena la red 2→2→1 con SGD (online) y guarda histórico detallado.
    Retorna pesos finales + historial de pérdidas y pesos.
    """
    np.random.seed(seed)
    W1 = np.random.randn(2, 2) * 0.5
    b1 = np.zeros(2)
    W2 = np.random.randn(1, 2) * 0.5
    b2 = np.zeros(1)

    historial_loss  = []
    historial_pesos = []

    for epoca in range(epocas):
        loss_epoca = 0.0

        for i in range(len(X)):
            x  = X[i]
            y  = float(Y[i].flat[0])   # escalar Python, evita acumulación de arrays

            # Forward
            z1   = W1 @ x + b1
            h    = sigmoid(z1)
            z2   = W2 @ h + b2
            yhat = float(sigmoid(z2).flat[0])  # escalar

            # Pérdida
            loss_epoca += 0.5 * (yhat - y) ** 2

            # Backward
            delta2 = (yhat - y) * yhat * (1 - yhat)   # escalar Python
            dW2    = delta2 * h.reshape(1, -1)
            db2    = delta2

            dL_dh  = W2.T.flatten() * delta2
            delta1 = dL_dh * h * (1 - h)
            dW1    = np.outer(delta1, x)
            db1_g  = delta1

            # Actualización
            W2 -= lr * dW2
            b2 -= lr * db2
            W1 -= lr * dW1
            b1 -= lr * db1_g

        loss_media = loss_epoca / len(X)
        historial_loss.append(loss_media)

        if epoca % guardar_cada == 0:
            historial_pesos.append({
                'epoca': epoca,
                'W1': W1.copy(), 'b1': b1.copy(),
                'W2': W2.copy(), 'b2': b2.copy(),
                'loss': loss_media
            })

    return W1, b1, W2, b2, historial_loss, historial_pesos

# Entrenar
W1_f, b1_f, W2_f, b2_f, losses, hist_pesos = entrenar(
    X, Y, epocas=10000, lr=0.5, seed=42)

print(f"Pérdida final: {losses[-1]:.6f}")
print(f"\nPesos finales:")
print(f"  W⁽¹⁾ =\n{W1_f}")
print(f"  b⁽¹⁾ = {b1_f}")
print(f"  W⁽²⁾ = {W2_f}")
print(f"  b⁽²⁾ = {b2_f}")

## 6. Visualización del Entrenamiento

In [ ]:
### 6.1 Curva de pérdida durante el entrenamiento

# Comparar diferentes tasas de aprendizaje
lrs = [0.1, 0.5, 1.0, 2.0]
colores_lr = ['#3498db', '#2ecc71', '#e67e22', '#e74c3c']

fig = go.Figure()
for lr_val, color in zip(lrs, colores_lr):
    _, _, _, _, ls, _ = entrenar(X, Y, epocas=10000, lr=lr_val, seed=42)
    epocas_plot = list(range(len(ls)))
    fig.add_trace(go.Scatter(
        x=epocas_plot, y=ls,
        mode='lines', name=f'η = {lr_val}',
        line=dict(color=color, width=2.5)
    ))

fig.update_layout(
    title=dict(text="Curva de pérdida: impacto de la tasa de aprendizaje η", font=dict(size=17)),
    xaxis=dict(title="Época", type='log'),
    yaxis=dict(title="Pérdida media L", type='log'),
    legend=dict(title="Tasa η"),
    width=800, height=430,
    plot_bgcolor='#f8f9fa'
)
fig.add_annotation(x=np.log10(200), y=np.log10(0.24),
                   text="η muy grande → inestabilidad",
                   font=dict(color='#e74c3c', size=11),
                   showarrow=False)
fig.show()

In [ ]:
### 6.2 Evolución de la frontera de decisión durante el entrenamiento

def calcular_frontera(W1, b1, W2, b2, resolucion=200):
    """Calcula la salida de la red en una malla 2D."""
    xx = np.linspace(-0.2, 1.2, resolucion)
    yy = np.linspace(-0.2, 1.2, resolucion)
    Z = np.zeros((resolucion, resolucion))
    for i, xi in enumerate(xx):
        for j, yj in enumerate(yy):
            x_ = np.array([xi, yj])
            z1_ = W1 @ x_ + b1
            h_  = sigmoid(z1_)
            z2_ = W2 @ h_ + b2
            Z[j, i] = sigmoid(z2_)[0]
    return xx, yy, Z

# Snapshots en épocas específicas
momentos = [0, 200, 500, 2000, 5000, 9950]
titulos  = [f"Época {hist_pesos[i]['epoca']}" for i in range(len(hist_pesos))
            if hist_pesos[i]['epoca'] in momentos]
snapshots = [p for p in hist_pesos if p['epoca'] in momentos]

# Garantizar exactamente 6 snapshots
snapshots_sel = []
for m in momentos:
    for p in hist_pesos:
        if p['epoca'] == m:
            snapshots_sel.append(p)
            break

fig = make_subplots(rows=2, cols=3,
                    subplot_titles=[f"Época {s['epoca']}  (L={s['loss']:.3f})"
                                    for s in snapshots_sel],
                    horizontal_spacing=0.05, vertical_spacing=0.12)

xor_x = X[:, 0]
xor_y_coord = X[:, 1]
xor_labels = Y.flatten()

for idx, snap in enumerate(snapshots_sel):
    row = idx // 3 + 1
    col = idx %  3 + 1

    xx, yy, Z = calcular_frontera(snap['W1'], snap['b1'], snap['W2'], snap['b2'], resolucion=80)

    fig.add_trace(go.Contour(
        x=xx, y=yy, z=Z,
        colorscale=[[0,'#fadbd8'],[0.5,'white'],[1,'#d5f5e3']],
        contours=dict(start=0, end=1, size=0.1,
                      coloring='fill', showlabels=False),
        line=dict(width=0), showscale=False,
        showlegend=False
    ), row=row, col=col)

    # Línea de decisión (ŷ = 0.5)
    fig.add_trace(go.Contour(
        x=xx, y=yy, z=Z,
        contours=dict(start=0.5, end=0.5, size=0.1,
                      coloring='lines'),
        line=dict(color='#2c3e50', width=2),
        showscale=False, showlegend=False
    ), row=row, col=col)

    # Puntos XOR
    for xi, yi, lab in zip(xor_x, xor_y_coord, xor_labels):
        fig.add_trace(go.Scatter(
            x=[xi], y=[yi],
            mode='markers',
            marker=dict(size=14,
                        color='#e74c3c' if lab == 0 else '#2ecc71',
                        symbol='circle' if lab == 0 else 'square',
                        line=dict(width=2, color='white')),
            showlegend=False
        ), row=row, col=col)

fig.update_layout(
    height=600, width=900,
    title_text="Evolución de la frontera de decisión (ŷ = 0.5)",
    title_font_size=17,
    plot_bgcolor='white', paper_bgcolor='white'
)
for i in range(1, 7):
    r, c = (i-1)//3+1, (i-1)%3+1
    fig.update_xaxes(range=[-0.2, 1.2], showticklabels=False, row=r, col=c)
    fig.update_yaxes(range=[-0.2, 1.2], showticklabels=False, row=r, col=c)
fig.show()

In [ ]:
### 6.3 Mapa de calor final de la red entrenada

xx_f, yy_f, Z_f = calcular_frontera(W1_f, b1_f, W2_f, b2_f, resolucion=250)

fig = go.Figure()

# Heatmap de probabilidad
fig.add_trace(go.Heatmap(
    x=xx_f, y=yy_f, z=Z_f,
    colorscale='RdYlGn',
    zmin=0, zmax=1,
    colorbar=dict(title="ŷ (probabilidad XOR=1)", thickness=15)
))

# Frontera de decisión
fig.add_trace(go.Contour(
    x=xx_f, y=yy_f, z=Z_f,
    contours=dict(start=0.5, end=0.5, size=1, coloring='lines'),
    line=dict(color='black', width=3, dash='dash'),
    showscale=False, name='Frontera de decisión'
))

# Puntos XOR
for xi, yi, lab in zip(X[:, 0], X[:, 1], Y.flatten()):
    txt = f"({int(xi)},{int(yi)}) → {int(lab)}"
    fig.add_trace(go.Scatter(
        x=[xi], y=[yi], mode='markers+text',
        marker=dict(size=24,
                    color='white',
                    symbol='circle' if lab == 0 else 'square',
                    line=dict(width=3, color='black')),
        text=[txt], textposition='top center',
        textfont=dict(size=13, color='black'),
        showlegend=False
    ))

fig.update_layout(
    title=dict(text="Red entrenada: mapa de probabilidades sobre el espacio de entrada", font=dict(size=16)),
    xaxis=dict(title="x₁", range=[-0.2, 1.2]),
    yaxis=dict(title="x₂", range=[-0.2, 1.2]),
    width=680, height=560,
)
fig.show()

In [ ]:
### 6.4 Evolución de los pesos durante el entrenamiento

# Extraer series temporales de cada peso
epocas_hist = [p['epoca'] for p in hist_pesos]

W1_00 = [p['W1'][0,0] for p in hist_pesos]
W1_01 = [p['W1'][0,1] for p in hist_pesos]
W1_10 = [p['W1'][1,0] for p in hist_pesos]
W1_11 = [p['W1'][1,1] for p in hist_pesos]
W2_00 = [p['W2'][0,0] for p in hist_pesos]
W2_01 = [p['W2'][0,1] for p in hist_pesos]
b1_0  = [p['b1'][0]   for p in hist_pesos]
b1_1  = [p['b1'][1]   for p in hist_pesos]
b2_0  = [p['b2'][0]   for p in hist_pesos]

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=("Pesos W⁽¹⁾ (capa oculta)",
                                    "Pesos W⁽²⁾ y sesgos b"))

# W1
for vals, nombre, color in [
    (W1_00, 'w¹₁₁ (h₁←x₁)', '#3498db'),
    (W1_01, 'w¹₁₂ (h₁←x₂)', '#2980b9'),
    (W1_10, 'w¹₂₁ (h₂←x₁)', '#e74c3c'),
    (W1_11, 'w¹₂₂ (h₂←x₂)', '#c0392b'),
    (b1_0,  'b¹₁',            '#9b59b6'),
    (b1_1,  'b¹₂',            '#8e44ad'),
]:
    fig.add_trace(go.Scatter(x=epocas_hist, y=vals,
        mode='lines', name=nombre,
        line=dict(color=color, width=2)), row=1, col=1)

# W2
for vals, nombre, color in [
    (W2_00, 'w²₁ (ŷ←h₁)',  '#2ecc71'),
    (W2_01, 'w²₂ (ŷ←h₂)',  '#27ae60'),
    (b2_0,  'b²',           '#f39c12'),
]:
    fig.add_trace(go.Scatter(x=epocas_hist, y=vals,
        mode='lines', name=nombre,
        line=dict(color=color, width=2)), row=1, col=2)

fig.update_layout(
    height=380, width=900,
    title_text="Evolución de pesos durante el entrenamiento",
    title_font_size=17,
    legend=dict(font=dict(size=10)),
    plot_bgcolor='#f8f9fa'
)
fig.update_xaxes(title_text="Época")
fig.update_yaxes(title_text="Valor del peso")
fig.show()

## 7. Verificación y Análisis de la Red Entrenada

In [ ]:
### 7.1 Tabla de predicciones finales

print("=" * 62)
print(f"{'x₁':>4} {'x₂':>4} | {'y':>6} | {'ŷ':>8} | {'Redondeado':>10} | {'✓/✗':>4}")
print("-" * 62)

todos_correctos = True
for i in range(len(X)):
    x     = X[i]
    y_val = int(Y[i].flat[0])          # escalar: Y[i] tiene shape (1,)
    z1_   = W1_f @ x + b1_f
    h_    = sigmoid(z1_)
    z2_   = W2_f @ h_ + b2_f
    yhat_ = float(sigmoid(z2_).flat[0])
    pred  = round(yhat_)
    ok    = "✓" if pred == y_val else "✗"
    if pred != y_val:
        todos_correctos = False
    print(f"{int(x[0]):>4} {int(x[1]):>4} | {y_val:>6} | {yhat_:>8.4f} | {pred:>10} | {ok:>4}")

print("=" * 62)
print(f"\n{'✅ Todas las predicciones son correctas!' if todos_correctos else '❌ Hay errores'}")

# Gráfica de barras comparando y vs ŷ
etiq_x = ["(0,0)→0", "(0,1)→1", "(1,0)→1", "(1,1)→0"]
preds_finales = []
for i in range(len(X)):
    x   = X[i]
    z1_ = W1_f @ x + b1_f
    h_  = sigmoid(z1_)
    z2_ = W2_f @ h_ + b2_f
    preds_finales.append(float(sigmoid(z2_).flat[0]))

fig = go.Figure()
fig.add_trace(go.Bar(
    x=etiq_x, y=Y.flatten(),
    name='y real', marker_color=['#e74c3c','#2ecc71','#2ecc71','#e74c3c'],
    opacity=0.6, width=0.35, offset=-0.2
))
fig.add_trace(go.Bar(
    x=etiq_x, y=preds_finales,
    name='ŷ predicho', marker_color=['#c0392b','#27ae60','#27ae60','#c0392b'],
    opacity=0.9, width=0.35, offset=0.05
))
fig.add_hline(y=0.5, line_dash='dot', line_color='gray',
              annotation_text='umbral de decisión (0.5)')

fig.update_layout(
    title="Predicciones finales vs valores reales",
    yaxis=dict(title="Valor", range=[0, 1.1]),
    xaxis=dict(title="Muestra"),
    barmode='overlay',
    width=650, height=380,
    plot_bgcolor='#f8f9fa',
    legend=dict(x=0.7, y=0.95)
)
fig.show()

### 7.2 Visualización de las representaciones internas (espacio oculto)



Una manera poderosa de entender por qué la red resuelve XOR es ver **cómo transforma las entradas** en el espacio de las neuronas ocultas $(h_1, h_2)$. En ese espacio, los puntos SÍ son linealmente separables.


In [ ]:

# Calcular activaciones ocultas para los 4 puntos XOR
print("Representaciones en el espacio oculto:")
print(f"{'Entrada':^12} | {'y':>4} | {'h₁':>8} | {'h₂':>8}")
print("-" * 40)
ocultas = []
for i in range(len(X)):
    x     = X[i]
    y_val = int(Y[i].flat[0])          # escalar: Y[i] tiene shape (1,)
    z1_   = W1_f @ x + b1_f
    h_    = sigmoid(z1_)
    ocultas.append(h_)
    print(f"({int(x[0])},{int(x[1])})       | {y_val:>4} | {h_[0]:>8.4f} | {h_[1]:>8.4f}")

ocultas = np.array(ocultas)

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=("Espacio de entrada (no separable)",
                                    "Espacio oculto (linealmente separable!)"))

# Espacio de entrada
for xi, yi, lab in zip(X[:,0], X[:,1], Y.flatten()):
    fig.add_trace(go.Scatter(
        x=[xi], y=[yi], mode='markers+text',
        marker=dict(size=22, color='#e74c3c' if lab==0 else '#2ecc71',
                    symbol='circle' if lab==0 else 'square',
                    line=dict(width=2, color='white')),
        text=[f"({int(xi)},{int(yi)})"],
        textposition='top center', showlegend=False
    ), row=1, col=1)

# Espacio oculto
for i, (h_, lab) in enumerate(zip(ocultas, Y.flatten())):
    xi_, yi_ = X[i]
    fig.add_trace(go.Scatter(
        x=[h_[0]], y=[h_[1]], mode='markers+text',
        marker=dict(size=22, color='#e74c3c' if lab==0 else '#2ecc71',
                    symbol='circle' if lab==0 else 'square',
                    line=dict(width=2, color='white')),
        text=[f"({int(xi_)},{int(yi_)})"],
        textposition='top center', showlegend=False
    ), row=1, col=2)

# Línea separadora en espacio oculto: w₁h₁ + w₂h₂ + b = 0  →  h₂ = -(w₁h₁+b)/w₂
h_vals = np.linspace(ocultas[:,0].min()-0.05, ocultas[:,0].max()+0.05, 100)
w1_out, w2_out = float(W2_f[0,0]), float(W2_f[0,1])
b_out = float(b2_f.flat[0])
if abs(w2_out) > 1e-6:
    h2_sep = -(w1_out * h_vals + b_out) / w2_out
    fig.add_trace(go.Scatter(
        x=h_vals, y=h2_sep,
        mode='lines', line=dict(color='#2c3e50', width=2.5, dash='dash'),
        name='Frontera lineal', showlegend=True
    ), row=1, col=2)

fig.update_layout(
    height=420, width=900,
    title_text="La magia de la capa oculta: transforma el espacio de representación",
    title_font_size=15,
    plot_bgcolor='#f8f9fa'
)
fig.update_xaxes(title_text="x₁", row=1, col=1)
fig.update_yaxes(title_text="x₂", row=1, col=1)
fig.update_xaxes(title_text="h₁ (neurona oculta 1)", row=1, col=2)
fig.update_yaxes(title_text="h₂ (neurona oculta 2)", row=1, col=2)
fig.show()

print("\n¡En el espacio oculto los puntos SÍ son linealmente separables!")

## 8. Conclusiones y Puntos Clave

### Lo que aprendimos

1. **XOR no es linealmente separable** → un perceptrón simple nunca puede aprenderlo. Necesitamos al menos **una capa oculta** para crear representaciones no lineales del espacio de entrada.

2. **La capa oculta es una transformación de representación**: los puntos que no son separables en el espacio original $(x_1, x_2)$ **sí lo son** en el espacio oculto $(h_1, h_2)$. La red "aprende" una representación útil.

3. **Backpropagation = regla de la cadena aplicada sistemáticamente**:

$$\frac{\partial \mathcal{L}}{\partial w^{(1)}_{jk}} = \underbrace{\delta^{(2)}}_{\text{error salida}} \cdot \underbrace{w^{(2)}_j}_{\text{peso conexión}} \cdot \underbrace{\sigma'(z^{(1)}_j)}_{\text{sensibilidad}} \cdot \underbrace{x_k}_{\text{entrada}}$$

4. **La tasa de aprendizaje $\eta$ importa**: muy pequeña → convergencia lenta; muy grande → inestabilidad u oscilación.

5. **El factor $\frac{1}{2}$ en la pérdida** es una convención que simplifica cálculos; el mínimo es el mismo.

---

### Preguntas para reflexionar

- ¿Qué pasaría si usáramos una función de activación **lineal** en la capa oculta? ¿Podría la red aprender XOR?
- ¿Cuántas neuronas ocultas son el mínimo necesario para XOR? ¿Por qué?
- ¿Por qué los pesos iniciales no deben ser todos cero? (Pista: piensa en el problema de la simetría)
- ¿Qué pasa con el gradiente de la sigmoide cuando $|z|$ es muy grande? (Problema del gradiente evanescente)

---

### Extensiones naturales

| Concepto | Descripción |
|----------|-------------|
| **Función de pérdida Binary Cross-Entropy** | Más apropiada para clasificación binaria que MSE |
| **Batch Gradient Descent** | Promediar gradientes de todas las muestras antes de actualizar |
| **ReLU como activación** | Alivia el problema del gradiente evanescente |
| **Momentum / Adam** | Optimizadores más sofisticados que GD puro |
| **Regularización** | L2 / Dropout para evitar sobreajuste |

---

*Este notebook fue diseñado como base didáctica para entender backpropagation desde sus fundamentos. Todo el código es desde cero, sin frameworks de deep learning.*